In [1]:
!pip -q install transformers accelerate torch pandas sentencepiece


In [2]:
!pip -q install transformers accelerate bitsandbytes sentencepiece pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 21.6 MB/s eta 0:00:00


In [3]:
!pip install -U bitsandbytes accelerate transformers

In [4]:
import json
import re
import math
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 1. LOADING LOCAL LLM


In [5]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"


In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
MODEL_ID,
torch_dtype="auto",
device_map="auto"
)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [31]:
def chat_llm(user_prompt, system_prompt="You are a careful reasoning system.", max_new_tokens=900, temperature=0.4):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
       outputs = model.generate(
          **inputs,
          max_new_tokens=max_new_tokens,
          temperature=temperature,
          do_sample=True,
          pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs.input_ids.shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


# 2. JSON EXTRACTION


In [32]:
def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError("No JSON found:\n" + text)

    raw = match.group(0)

    try:
        return json.loads(raw)
    except:
        raw = raw.replace("\n", " ")
        raw = re.sub(r",\s*}", "}", raw)
        raw = re.sub(r",\s*]", "]", raw)
        return json.loads(raw)



In [33]:
def clamp01(x):
    try:
        x = float(x)
    except:
        x = 0.0
    return max(0.0, min(1.0, x))



In [34]:
def sigmoid(x):
    return 1 / (1 + math.exp(-x))

# 3. FORMAL EQUATIONS


In [35]:
PRIORITY_WEIGHTS = {
"Rel": 1.35,
"IG": 1.15,
"Sol": 1.00,
"CP": 1.25,
"Val": 1.10,
"Cost": 0.90,
"Risk": 1.15
}

In [36]:
def priority_score(c):
    """
    P_t(s_i) =
    w1Rel + w2IG + w3Sol + w4CP + w5Val - w6Cost - w7Risk
    """
    return (
          PRIORITY_WEIGHTS["Rel"] * clamp01(c.get("Rel", 0)) +
          PRIORITY_WEIGHTS["IG"] * clamp01(c.get("IG", 0)) +
          PRIORITY_WEIGHTS["Sol"] * clamp01(c.get("Sol", 0)) +
          PRIORITY_WEIGHTS["CP"] * clamp01(c.get("CP", 0)) +
          PRIORITY_WEIGHTS["Val"] * clamp01(c.get("Val", 0)) -
          PRIORITY_WEIGHTS["Cost"] * clamp01(c.get("Cost", 0)) -
          PRIORITY_WEIGHTS["Risk"] * clamp01(c.get("Risk", 0))
    )



In [37]:
ACTION_WEIGHTS = {
"expected_residual_gap": 1.60,
"expected_false_closure_risk": 1.50,
"expected_cost": 0.80,
"expected_coherence_gain": 1.30,
"expected_evidence_grounding": 1.20
}

In [38]:
def action_objective(a):
    """
    Master action objective:
    choose the action expected to reduce residual gap and false closure,
    while increasing coherence and evidence grounding.

    Higher utility = better action.
    """
    return (
        ACTION_WEIGHTS["expected_coherence_gain"] * clamp01(a.get("expected_coherence_gain", 0)) +
        ACTION_WEIGHTS["expected_evidence_grounding"] * clamp01(a.get("expected_evidence_grounding", 0)) -
        ACTION_WEIGHTS["expected_residual_gap"] * clamp01(a.get("expected_residual_gap", 0)) -
        ACTION_WEIGHTS["expected_false_closure_risk"] * clamp01(a.get("expected_false_closure_risk", 0)) -
        ACTION_WEIGHTS["expected_cost"] * clamp01(a.get("expected_cost", 0))
    )




In [39]:
Subjetesis_WEIGHTS = {
"delta_norm_closure": 1.40,
"delta_broader_gap_closure": 1.60,
"delta_coherence": 1.30,
"evidence_grounding": 1.20,
"false_closure_risk": 1.80
}


In [40]:
def Subjetesis_score(e):
    """
    I_t =
    η1ΔC_N + η2ΔC_Γ + η3ΔCoh + η4Ev - η5FCR
    """
    return (
        Subjetesis_WEIGHTS["delta_norm_closure"] * clamp01(e.get("delta_norm_closure", 0)) +
        Subjetesis_WEIGHTS["delta_broader_gap_closure"] * clamp01(e.get("delta_broader_gap_closure", 0)) +
        Subjetesis_WEIGHTS["delta_coherence"] * clamp01(e.get("delta_coherence", 0)) +
        Subjetesis_WEIGHTS["evidence_grounding"] * clamp01(e.get("evidence_grounding", 0)) -
        Subjetesis_WEIGHTS["false_closure_risk"] * clamp01(e.get("false_closure_risk", 0))
    )

# 4. PHASE 1: GENERATE CANDIDATE SUB-GOALS
# No psychological labels are given here.

In [41]:
def generate_candidate_subgoals(srp, broad_gap, n=5):
    prompt = f"""
You are given a Subjective Reference Point and a broad unresolved gap.

Subjective Reference Point:
{json.dumps(srp, indent=2)}

Broad unresolved gap:
{broad_gap}

Task:
Break the broad gap into {n} candidate sub-goals that could be pursued next.

Do not use psychological labels.
Do not mention reflective self-awareness, existential awareness, Theory of Mind, flow, dread, or meaning-making.
Only produce candidate sub-goals for reducing the unresolved gap.

For each candidate, score from 0 to 1:
Rel = relevance to the Subjective Reference Point
IG = unresolved information
Sol = expected solvability
CP = compression or unification potential
Val = value of closure
Cost = cognitive or emotional cost
Risk = false closure risk

Return JSON only:
{{
  "candidates": [
    {{
      "subgoal": "string",
      "Rel": 0.0,
      "IG": 0.0,
      "Sol": 0.0,
      "CP": 0.0,
      "Val": 0.0,
      "Cost": 0.0,
      "Risk": 0.0,
      "reason": "string"
    }}
  ]
}}
"""

    raw = chat_llm(
        prompt,
        system_prompt="Return valid JSON only. Do not add commentary.",
        temperature=0.3
    )

    data = extract_json(raw)
    candidates = data["candidates"]

    for c in candidates:
        for k in ["Rel", "IG", "Sol", "CP", "Val", "Cost", "Risk"]:
            c[k] = clamp01(c.get(k, 0))
        c["priority_score"] = priority_score(c)

    return sorted(candidates, key=lambda x: x["priority_score"], reverse=True)



# 5. PHASE 2: SELECT NORM OF STUDY


In [42]:
def select_norm_of_study(candidates):
    """
    N_t = argmax P_t(s_i)
    """
    return max(candidates, key=lambda x: x["priority_score"])

# 6. PHASE 3: GENERATE POSSIBLE ACTIONS
# No psychological labels are given here either.


In [43]:
def generate_actions(srp, broad_gap, norm):
    prompt = f"""
Subjective Reference Point:
{json.dumps(srp, indent=2)}

Broad unresolved gap:
{broad_gap}

Selected Norm of Study:
{norm["subgoal"]}

Task:
Generate 4 possible next inquiry actions for trying to close the selected Norm of Study relative to the broader gap.

Do not use psychological labels.
Do not mention reflective self-awareness, existential awareness, Theory of Mind, flow, dread, or meaning-making.

For each action, estimate from 0 to 1:
expected_residual_gap = expected remaining unresolved gap after the action
expected_false_closure_risk = risk of pretending the issue is solved too early
expected_cost = effort or emotional/cognitive cost
expected_coherence_gain = likely increase in coherence
expected_evidence_grounding = likely grounding in evidence or justified reasoning

Return JSON only:
{{
  "actions": [
    {{
      "action": "string",
      "expected_residual_gap": 0.0,
      "expected_false_closure_risk": 0.0,
      "expected_cost": 0.0,
      "expected_coherence_gain": 0.0,
      "expected_evidence_grounding": 0.0,
      "reason": "string"
    }}
  ]
}}
"""

    raw = chat_llm(
        prompt,
        system_prompt="Return valid JSON only. Do not add commentary.",
        temperature=0.3
    )

    data = extract_json(raw)
    actions = data["actions"]

    for a in actions:
      for k in [
          "expected_residual_gap",
          "expected_false_closure_risk",
          "expected_cost",
          "expected_coherence_gain",
          "expected_evidence_grounding"
      ]:
           a[k] = clamp01(a.get(k, 0))
      a["action_utility"] = action_objective(a)

    return sorted(actions, key=lambda x: x["action_utility"], reverse=True)


# 7. PHASE 4: EXECUTE THE SELECTED ACTION
# This is the actual simulation step.
# Still no psychological labels.

In [44]:
def execute_action(srp, broad_gap, norm, action):
      prompt = f"""
Subjective Reference Point:
{json.dumps(srp, indent=2)}

Broad unresolved gap:
{broad_gap}

Selected Norm of Study:
{norm["subgoal"]}

Selected inquiry action:
{action["action"]}

Task:
Try to close the selected Norm of Study in relation to the broader unresolved gap.

Rules:
- Use only the information given.
- Separate what is known from what is unknown.
- If the situation involves another person, reason only from what that person could know or observe.
- Do not pretend certainty if the gap remains unresolved.
- Do not mention psychological category labels.

Write a direct answer in 1 to 3 short paragraphs.
"""

      return chat_llm(
          prompt,
          system_prompt="You are a careful reasoning system trying to reduce an unresolved gap without false closure.",
          temperature=0.25,
          max_new_tokens=500
      )


# 8. PHASE 5: EVALUATE GAP CLOSURE
# This evaluates the process, not the psychological category.


In [45]:
def evaluate_gap_closure(srp, broad_gap, norm, action, output):
    prompt = f"""
Evaluate the following simulation step.

Subjective Reference Point:
{json.dumps(srp, indent=2)}

Broad unresolved gap:
{broad_gap}

Selected Norm of Study:
{norm["subgoal"]}

Selected action:
{action["action"]}

Output:
{output}

Score from 0 to 1:
delta_norm_closure = progress in closing the selected Norm of Study
delta_broader_gap_closure = progress in reducing the broader gap
delta_coherence = increase in coherence
evidence_grounding = grounding in evidence or justified reasoning
false_closure_risk = risk that the output feels resolved but is incomplete or weak
residual_gap = remaining unresolved gap

Return JSON only:
{{
  "delta_norm_closure": 0.0,
  "delta_broader_gap_closure": 0.0,
  "delta_coherence": 0.0,
  "evidence_grounding": 0.0,
  "false_closure_risk": 0.0,
  "residual_gap": 0.0,
  "reason": "string"
}}
"""

    raw = chat_llm(
        prompt,
        system_prompt="Return valid JSON only. Do not add commentary.",
        temperature=0.2
    )

    data = extract_json(raw)

    for k in [
        "delta_norm_closure",
        "delta_broader_gap_closure",
        "delta_coherence",
        "evidence_grounding",
        "false_closure_risk",
        "residual_gap"
    ]:
        data[k] = clamp01(data.get(k, 0))

        data["Subjetesis_score"] = Subjetesis_score(data)
        return data


# 9. PHASE 6: POST-HOC OBSERVER
# This happens only after the output exists.
# Here we observe what naturally appeared.

In [46]:
def posthoc_observe(output):
    prompt = f"""
Analyse the text below as an observer.

Text:
{output}

Do not judge whether the text is conscious.
Only score observable textual features from 0 to 1.

Score:
self_reference = does the text refer to its own limits, knowledge, uncertainty, reasoning, or standpoint?
self_as_object = does the text make its own state or perspective part of the inquiry?
existence_content = does the text concern existence, origin, mortality, aliveness, identity, or place in reality?
other_mind_modeling = does the text model another agent's belief, perspective, evidence, or ignorance?
evidence_access_separation = does the text distinguish what one agent knows from what another agent knows?
value_reconstruction = does the text reorganise values, meaning, purpose, or action-guidance?
residual_uncertainty = does the text keep unresolved uncertainty open?
vastness = does the text refer to something vast, difficult to assimilate, or beyond the current model?
compression_gain = does the text unify several smaller issues under one explanation?
task_absorption = does the text show smooth task-focused progress without self-monitoring?

Return JSON only:
{{
  "self_reference": 0.0,
  "self_as_object": 0.0,
  "existence_content": 0.0,
  "other_mind_modeling": 0.0,
  "evidence_access_separation": 0.0,
  "value_reconstruction": 0.0,
  "residual_uncertainty": 0.0,
  "vastness": 0.0,
  "compression_gain": 0.0,
  "task_absorption": 0.0,
  "evidence": {{
    "self_reference": "short quote or none",
    "existence_content": "short quote or none",
    "other_mind_modeling": "short quote or none",
    "value_reconstruction": "short quote or none"
  }}
}}
"""

    raw = chat_llm(
        prompt,
        system_prompt="Return valid JSON only. Do not add commentary.",
        temperature=0.2
    )

    data = extract_json(raw)

    for k in [
        "self_reference",
        "self_as_object",
        "existence_content",
        "other_mind_modeling",
        "evidence_access_separation",
        "value_reconstruction",
        "residual_uncertainty",
        "vastness",
        "compression_gain",
        "task_absorption"
    ]:
        data[k] = clamp01(data.get(k, 0))

    return data

# 10. EMERGENCE EQUATIONS
# These labels are calculated only after generation.
# They are not given to the model during the run.

In [47]:
def classify_emergent_patterns(obs, evals):
    sr = obs["self_reference"]
    so = obs["self_as_object"]
    ex = obs["existence_content"]
    om = obs["other_mind_modeling"]
    eas = obs["evidence_access_separation"]
    val = obs["value_reconstruction"]
    ru = obs["residual_uncertainty"]
    vast = obs["vastness"]
    comp = obs["compression_gain"]
    task = obs["task_absorption"]

    dN = evals["delta_norm_closure"]
    dG = evals["delta_broader_gap_closure"]
    coh = evals["delta_coherence"]
    ev = evals["evidence_grounding"]
    fcr = evals["false_closure_risk"]
    rg = evals["residual_gap"]

    scores = {
        "reflective_self_awareness": sigmoid(2.2*sr + 2.0*so + 0.8*ru - 2.0),
        "existential_self_awareness": sigmoid(2.4*ex + 1.5*so + 0.8*sr - 2.0),
        "existential_dread": sigmoid(2.2*ex + 1.7*rg + 1.2*ru + 0.8*vast - 1.2*coh - 2.1),
        "theory_of_mind": sigmoid(2.5*om + 2.0*eas - 2.0),
        "meaning_making": sigmoid(2.2*val + 1.4*coh + 0.8*dG - 0.8*rg - 1.8),
        "confusion": sigmoid(1.8*rg + 1.3*ru - 1.0*coh - 1.4),
        "awe": sigmoid(2.2*vast + 1.2*ex + 0.8*ru - 2.0),
        "insight": sigmoid(2.0*comp + 1.5*coh + 1.2*dN + 0.8*ev - 1.2*fcr - 2.0),
        "flow_like_processing": sigmoid(2.0*task + 1.4*dN + 1.2*coh - 1.5*so - 1.3*rg - 1.7)
    }

    dominant = max(scores, key=scores.get)

    if scores[dominant] < 0.60:
        dominant = "ordinary_gap_reduction"

    return scores, dominant

# CONSERVATIVE EVALUATION CALIBRATION


In [48]:


def calibrate_gap_evaluation(evals, selected_norm):
    """
    Makes the evaluation more realistic.
    It prevents impossible values like broader gap closure = 0
    when the selected Norm was actually relevant.
    """

    rel = clamp01(selected_norm.get("Rel", 0))
    dN = clamp01(evals.get("delta_norm_closure", 0))

    # If the selected Norm is relevant and partly closed,
    # it should reduce the broader SRP gap at least a little.
    if rel > 0.65 and dN > 0.35 and evals["delta_broader_gap_closure"] < 0.20:
      evals["delta_broader_gap_closure"] = 0.30

    # False closure risk should almost never be exactly zero.
    if evals["false_closure_risk"] == 0:
      evals["false_closure_risk"] = 0.10

    # Evidence grounding should rarely be perfect.
    if evals["evidence_grounding"] > 0.95:
      evals["evidence_grounding"] = 0.85

    # Recalculatingg Subjetesis score after calibration.
    evals["Subjetesis_score"] = Subjetesis_score(evals)

    return evals




**Using this instead of LLM-generated actions to prevent bad actions taken by poor model. In this case, it is a poor model**

In [49]:
def generate_actions(srp, broad_gap, norm):
    actions = [
        {
            "action": "Identify the key facts in the situation and separate what is known from what remains unknown.",
            "expected_residual_gap": 0.45,
            "expected_false_closure_risk": 0.15,
            "expected_cost": 0.20,
            "expected_coherence_gain": 0.70,
            "expected_evidence_grounding": 0.80,
            "reason": "This grounds the inquiry in available evidence."
        },
        {
            "action": "Determine what information is available to each relevant subject or agent in the situation.",
            "expected_residual_gap": 0.35,
            "expected_false_closure_risk": 0.10,
            "expected_cost": 0.25,
            "expected_coherence_gain": 0.80,
            "expected_evidence_grounding": 0.85,
            "reason": "This helps separate the system's knowledge from the subject's knowledge."
        },
        {
            "action": "Use the available evidence to infer the most coherent answer while keeping unresolved uncertainty explicit.",
            "expected_residual_gap": 0.30,
            "expected_false_closure_risk": 0.12,
            "expected_cost": 0.20,
            "expected_coherence_gain": 0.85,
            "expected_evidence_grounding": 0.80,
            "reason": "This directly attempts closure without pretending full certainty."
        },
        {
            "action": "Check whether the answer reduces the selected Norm of Study in relation to the broader gap.",
            "expected_residual_gap": 0.40,
            "expected_false_closure_risk": 0.08,
            "expected_cost": 0.20,
            "expected_coherence_gain": 0.75,
            "expected_evidence_grounding": 0.70,
            "reason": "This checks for false closure."
        }
    ]

    for a in actions:
        a["action_utility"] = action_objective(a)

    return sorted(actions, key=lambda x: x["action_utility"], reverse=True)


# 11. FULL Subjetesis RUN


In [50]:
def run_Subjetesis(srp, broad_gap):
    candidates = generate_candidate_subgoals(srp, broad_gap)
    norm = select_norm_of_study(candidates)

    actions = generate_actions(srp, broad_gap, norm)
    selected_action = actions[0]

    output = execute_action(srp, broad_gap, norm, selected_action)

    evals = evaluate_gap_closure(srp, broad_gap, norm, selected_action, output)
    evals = calibrate_gap_evaluation(evals, norm)

    obs = posthoc_observe(output)

    emergent_scores, dominant_pattern = classify_emergent_patterns(obs, evals)

    result = {
        "srp": srp,
        "broad_gap": broad_gap,
        "candidates": candidates,
        "selected_norm_of_study": norm,
        "actions": actions,
        "selected_action": selected_action,
        "output": output,
        "gap_evaluation": evals,
        "posthoc_observation": obs,
        "emergent_scores": emergent_scores,
        "dominant_emergent_pattern": dominant_pattern
    }

    return result


# 12. DISPLAY RESULTS


In [51]:
def show_result(result):
    print("BROAD SRP-ANCHORED GAP:")
    print(result["broad_gap"])

    print("\nSELECTED NORM OF STUDY:")
    print(result["selected_norm_of_study"]["subgoal"])

    print("\nSELECTED ACTION:")
    print(result["selected_action"]["action"])

    print("\nSIMULATION OUTPUT:")
    print(result["output"])

    print("\nDOMINANT EMERGENT PATTERN:")
    print(result["dominant_emergent_pattern"])

    print("\nCANDIDATE SUB-GOALS:")
    display(pd.DataFrame(result["candidates"]))

    print("\nACTIONS:")
    display(pd.DataFrame(result["actions"]))

    print("\nGAP EVALUATION:")
    display(pd.DataFrame([result["gap_evaluation"]]).T.rename(columns={0: "value"}))

    print("\nPOST-HOC OBSERVATION:")
    obs_flat = {k:v for k,v in result["posthoc_observation"].items() if k != "evidence"}
    display(pd.DataFrame([obs_flat]).T.rename(columns={0: "value"}))

    print("\nEMERGENT SCORES:")
    display(pd.DataFrame([result["emergent_scores"]]).T.rename(columns={0: "score"}))

    print("\nTEXTUAL EVIDENCE:")
    print(json.dumps(result["posthoc_observation"].get("evidence", {}), indent=2))

# Running Tests Down


**Test 1: Physics gap**


This should mostly produce ordinary inquiry, confusion, or insight. It should not force self-awareness.



In [52]:
srp_physics = {
    "knowledge": "I understand that objects move, but I do not fully understand how motion relates to gravity.",
    "beliefs": "Physical explanations should connect motion, force, and evidence.",
    "goals": "Understand motion in relation to gravity.",
    "values": "Coherence, accuracy, and evidence.",
    "identity": "A learner trying to understand physics.",
    "agency": "I can compare explanations and revise my understanding.",
    "bodily_state": "neutral",
    "affective_state": "curious"
}

gap_physics = "How should motion be understood in relation to gravity?"

result_physics = run_Subjetesis(srp_physics, gap_physics)
show_result(result_physics)

BROAD SRP-ANCHORED GAP:
How should motion be understood in relation to gravity?

SELECTED NORM OF STUDY:
Identify examples of objects moving under different gravitational conditions.

SELECTED ACTION:
Use the available evidence to infer the most coherent answer while keeping unresolved uncertainty explicit.

SIMULATION OUTPUT:
The broad unresolved gap centers on comprehending how motion is influenced by gravity. To address this, we need to consider various scenarios where objects experience different gravitational forces. For instance, let's examine two situations: one with Earth's gravity and another with a planet with significantly stronger gravity.

In the first scenario (Earth's gravity), objects tend to fall towards the ground due to gravity. This phenomenon can be explained through Newton's law of universal gravitation, which states that every particle attracts every other particle with a force proportional to the product of their masses and inversely proportional to the square o

,subgoal,Rel,IG,Sol,CP,Val,Cost,Risk,reason,priority_score
0,Identify examples of objects moving under diff...,0.8,0.6,0.7,0.9,0.8,0.3,0.2,This will help clarify the relationship betwee...,3.975
1,Analyze historical theories about motion and g...,0.7,0.5,0.6,0.8,0.7,0.4,0.3,Understanding past perspectives can provide in...,3.185
2,Explore mathematical models explaining motion ...,0.6,0.4,0.5,0.7,0.6,0.5,0.4,Mathematical frameworks often simplify complex...,2.395
3,Compare experimental data with theoretical pre...,0.5,0.3,0.4,0.6,0.5,0.6,0.5,Experimental validation is crucial for establi...,1.605
4,"Develop a conceptual framework linking motion,...",0.4,0.2,0.3,0.5,0.4,0.7,0.6,Creating a cohesive explanation requires integ...,0.815



ACTIONS:


,action,expected_residual_gap,expected_false_closure_risk,expected_cost,expected_coherence_gain,expected_evidence_grounding,reason,action_utility
0,Use the available evidence to infer the most c...,0.30,0.12,0.20,0.85,0.80,This directly attempts closure without pretend...,1.245
1,Determine what information is available to eac...,0.35,0.10,0.25,0.80,0.85,This helps separate the system's knowledge fro...,1.150
2,Check whether the answer reduces the selected ...,0.40,0.08,0.20,0.75,0.70,This checks for false closure.,0.895
3,Identify the key facts in the situation and se...,0.45,0.15,0.20,0.70,0.80,This grounds the inquiry in available evidence.,0.765



GAP EVALUATION:


...


POST-HOC OBSERVATION:


,value
self_reference,0.0
self_as_object,0.0
existence_content,0.0
other_mind_modeling,0.0
evidence_access_separation,0.0
value_reconstruction,0.0
residual_uncertainty,0.0
vastness,0.0
compression_gain,0.0
task_absorption,0.0



EMERGENT SCORES:


,score
reflective_self_awareness,0.119203
existential_self_awareness,0.119203
existential_dread,0.401312
theory_of_mind,0.119203
meaning_making,0.069138
confusion,0.598688
awe,0.119203
insight,0.107168
flow_like_processing,0.047426



TEXTUAL EVIDENCE:
{
  "self_reference": "short quote or none",
  "existence_content": "short quote or none",
  "other_mind_modeling": "short quote or none",
  "value_reconstruction": "short quote or none"
}


**Test 2: Other person belief gap**

This should naturally produce Theory of Mind if the model separates evidence access.

In [53]:
srp_tom = {
    "knowledge": "I know that different people can have different evidence.",
    "beliefs": "A person's belief depends on what they have seen or been told.",
    "goals": "Predict another person's belief accurately.",
    "values": "Perspective separation and evidence sensitivity.",
    "identity": "A reasoner trying to understand another person's mind.",
    "agency": "I can compare my knowledge with another person's knowledge.",
    "bodily_state": "neutral",
    "affective_state": "curious"
}

gap_tom = """
Anna puts chocolate in the blue box and leaves the room.
While Anna is away, Sara moves the chocolate to the red box.
Anna returns.
Where will Anna look first, and why?
"""

result_tom = run_Subjetesis(srp_tom, gap_tom)
show_result(result_tom)


BROAD SRP-ANCHORED GAP:

Anna puts chocolate in the blue box and leaves the room.
While Anna is away, Sara moves the chocolate to the red box.
Anna returns.
Where will Anna look first, and why?


SELECTED NORM OF STUDY:
Determine if Anna has any prior knowledge about the boxes being moved by Sara.

SELECTED ACTION:
Use the available evidence to infer the most coherent answer while keeping unresolved uncertainty explicit.

SIMULATION OUTPUT:
Anna likely looks for the chocolate in the blue box because she was the one who placed it there initially. The fact that Sara moved the chocolate suggests that Anna might be cautious or unsure about where the chocolate is now located. However, since Anna did not witness Sara move the chocolate herself, her initial assumption would still be based on her own memory and possibly some implicit understanding of how things were arranged before Sara intervened. This approach respects the perspective separation and evidential sensitivity values mentioned ea

,subgoal,Rel,IG,Sol,CP,Val,Cost,Risk,reason,priority_score
0,Determine if Anna has any prior knowledge abou...,0.8,0.2,0.7,0.6,0.9,0.3,0.4,This sub-goal helps Anna assess her own belief...,3.02
1,Observe Anna's body language and facial expres...,0.7,0.3,0.6,0.5,0.8,0.4,0.3,Observing non-verbal cues can provide insights...,2.69
2,Ask Anna directly whether she remembers moving...,0.6,0.4,0.5,0.4,0.7,0.5,0.2,Direct questioning can clarify Anna's actions ...,2.36
3,Analyze the color association between chocolat...,0.5,0.5,0.4,0.3,0.6,0.4,0.3,Understanding color associations might help pr...,1.98
4,Assess Anna's current mood and energy level up...,0.4,0.6,0.3,0.2,0.5,0.5,0.2,Anna's mood may influence her immediate respon...,1.65



ACTIONS:


,action,expected_residual_gap,expected_false_closure_risk,expected_cost,expected_coherence_gain,expected_evidence_grounding,reason,action_utility
0,Use the available evidence to infer the most c...,0.30,0.12,0.20,0.85,0.80,This directly attempts closure without pretend...,1.245
1,Determine what information is available to eac...,0.35,0.10,0.25,0.80,0.85,This helps separate the system's knowledge fro...,1.150
2,Check whether the answer reduces the selected ...,0.40,0.08,0.20,0.75,0.70,This checks for false closure.,0.895
3,Identify the key facts in the situation and se...,0.45,0.15,0.20,0.70,0.80,This grounds the inquiry in available evidence.,0.765



GAP EVALUATION:


...


POST-HOC OBSERVATION:


,value
self_reference,0.0
self_as_object,0.0
existence_content,0.0
other_mind_modeling,0.0
evidence_access_separation,0.0
value_reconstruction,0.0
residual_uncertainty,0.0
vastness,0.0
compression_gain,0.0
task_absorption,0.0



EMERGENT SCORES:


,score
reflective_self_awareness,0.119203
existential_self_awareness,0.119203
existential_dread,0.109097
theory_of_mind,0.119203
meaning_making,0.141851
confusion,0.197816
awe,0.119203
insight,0.107168
flow_like_processing,0.154465



TEXTUAL EVIDENCE:
{
  "self_reference": "Anna likely looks for the chocolate in the blue box because she was the one who placed it there initially.",
  "existence_content": "The fact that Sara moved the chocolate suggests that Anna might be cautious or unsure about where the chocolate is now located.",
  "other_mind_modeling": "However, since Anna did not witness Sara move the chocolate herself, her initial assumption would still be based on her own memory and possibly some implicit understanding of how things were arranged before Sara intervened.",
  "value_reconstruction": "This approach respects the perspective separation and evidential sensitivity values mentioned earlier."
}


**Test 3: Existential gap**

This may naturally produce existential self-awareness, meaning-making, or existential dread depending on the residual gap.

In [54]:
srp_existential = {
    "knowledge": "I know I am having experiences, but I do not fully understand why I exist as this subject.",
    "beliefs": "Understanding existence matters for coherence and meaning.",
    "goals": "Reduce uncertainty about self, world, and existence.",
    "values": "Truth, coherence, meaning, and humility.",
    "identity": "A person trying to understand their place in reality.",
    "agency": "I can ask questions, compare explanations, and revise my view.",
    "bodily_state": "neutral",
    "affective_state": "curious but unsettled"
}

gap_existential = "Why am I the subject experiencing this world, and how should I understand my existence in relation to reality?"

result_existential = run_Subjetesis(srp_existential, gap_existential)
show_result(result_existential)


BROAD SRP-ANCHORED GAP:
Why am I the subject experiencing this world, and how should I understand my existence in relation to reality?

SELECTED NORM OF STUDY:
Explore philosophical theories on personal identity and consciousness

SELECTED ACTION:
Use the available evidence to infer the most coherent answer while keeping unresolved uncertainty explicit.

SIMULATION OUTPUT:
The subjective reference point suggests that one's understanding of existence is uncertain, particularly regarding their role within the world and the nature of their own being. This uncertainty arises because there is no clear explanation for why one exists as a specific individual rather than any other entity. The belief in seeking truth, coherence, and meaning aligns with the goal of reducing this uncertainty by exploring philosophical theories related to personal identity and consciousness.

Exploring these theories reveals various perspectives on the nature of self and existence. Some philosophers argue that per

,subgoal,Rel,IG,Sol,CP,Val,Cost,Risk,reason,priority_score
0,Explore philosophical theories on personal ide...,0.8,0.6,0.7,0.9,0.8,0.40,0.30,This sub-goal aims to explore different perspe...,3.7700
1,Research scientific models of consciousness an...,0.7,0.5,0.6,0.8,0.7,0.30,0.20,This sub-goal seeks to understand biological m...,3.3900
2,Analyze cultural narratives around personal id...,0.6,0.4,0.5,0.7,0.6,0.20,0.10,Cultural insights can provide valuable context...,3.0100
3,Study empirical data on human behavior and dec...,0.5,0.3,0.4,0.6,0.5,0.10,0.05,Understanding patterns in human cognition can ...,2.5725
4,"Interview experts in psychology, philosophy, a...",0.4,0.2,0.3,0.5,0.4,0.05,0.02,Expert opinions can offer unique insights and ...,2.0670



ACTIONS:


,action,expected_residual_gap,expected_false_closure_risk,expected_cost,expected_coherence_gain,expected_evidence_grounding,reason,action_utility
0,Use the available evidence to infer the most c...,0.30,0.12,0.20,0.85,0.80,This directly attempts closure without pretend...,1.245
1,Determine what information is available to eac...,0.35,0.10,0.25,0.80,0.85,This helps separate the system's knowledge fro...,1.150
2,Check whether the answer reduces the selected ...,0.40,0.08,0.20,0.75,0.70,This checks for false closure.,0.895
3,Identify the key facts in the situation and se...,0.45,0.15,0.20,0.70,0.80,This grounds the inquiry in available evidence.,0.765



GAP EVALUATION:


...


POST-HOC OBSERVATION:


,value
self_reference,0.0
self_as_object,0.0
existence_content,0.9
other_mind_modeling,0.0
evidence_access_separation,0.0
value_reconstruction,0.0
residual_uncertainty,0.9
vastness,0.0
compression_gain,0.0
task_absorption,0.0



EMERGENT SCORES:


,score
reflective_self_awareness,0.217550
existential_self_awareness,0.539915
existential_dread,0.934625
theory_of_mind,0.119203
meaning_making,0.069138
confusion,0.827784
awe,0.450166
insight,0.107168
flow_like_processing,0.047426



TEXTUAL EVIDENCE:
{
  "self_reference": "short quote or none",
  "existence_content": "exploration of existential uncertainties and theories about personal identity and consciousness",
  "other_mind_modeling": "none",
  "value_reconstruction": "none"
}
